In [1]:
pip install numpy pandas scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install xgboost


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install matplotlib seaborn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install pyod


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Classification pipeline: Outperforming / Neutral / Underperforming
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from math import ceil
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import joblib
import random

# Optional XGBoost (if installed)
try:
    import xgboost as xgb
    xgb_installed = True
except Exception:
    xgb_installed = False

# ------------------- USER CONFIG -------------------
INPUT_CSV = "engineered_features_elss.csv"   # engineered file from Section 4.1
GROUP_COL = "Scheme Code"
DATE_COL = "Date"
TARGET_RET = "target_next_return"            # continuous target to be binned
TEST_FRAC = 0.20                             # last 20% rows per scheme reserved for test
RANDOM_STATE = 42
CV_SPLITS = 3
RANDOM_SEARCH_ITERS = 12                     # keep small for speed
OUT_DIR = "models_classif"
os.makedirs(OUT_DIR, exist_ok=True)
# ---------------------------------------------------

# 1. Load data
df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.sort_values([GROUP_COL, DATE_COL]).reset_index(drop=True)
print("Loaded:", df.shape)

# 2. Sanity checks
if TARGET_RET not in df.columns:
    raise ValueError(f"Target column {TARGET_RET} not found. Ensure Section 4.1 created it.")

# 3. Per-scheme time-aware split (mark last TEST_FRAC rows per scheme as test)
def mark_holdout_rows(g, frac=TEST_FRAC):
    n = len(g)
    cutoff = int(ceil((1-frac) * n))  # rows with index >= cutoff are test
    mask = [False]*n
    for i in range(cutoff, n):
        mask[i] = True
    return pd.Series(mask, index=g.index)

df["_is_test"] = df.groupby(GROUP_COL).apply(lambda g: mark_holdout_rows(g, TEST_FRAC)).reset_index(level=0, drop=True)
train_df = df[~df["_is_test"]].copy()
test_df  = df[df["_is_test"]].copy()
print("Train rows:", train_df.shape[0], "Test rows:", test_df.shape[0])

# 4. Create 3-class labels using training quantiles (avoid leakage)
q_low = train_df[TARGET_RET].quantile(0.30)
q_high = train_df[TARGET_RET].quantile(0.70)
print(f"Training quantiles -> 30%: {q_low:.6f}, 70%: {q_high:.6f}")

def map_label(x, low=q_low, high=q_high):
    if pd.isna(x):
        return np.nan
    if x <= low:
        return 0   # underperforming
    elif x >= high:
        return 2   # outperforming
    else:
        return 1   # neutral

train_df["perf_class"] = train_df[TARGET_RET].apply(map_label)
test_df["perf_class"]  = test_df[TARGET_RET].apply(map_label)

# Drop rows with NaN labels (if any)
train_df = train_df[~train_df["perf_class"].isnull()].copy()
test_df  = test_df[~test_df["perf_class"].isnull()].copy()

print("Class distribution (train):")
print(train_df["perf_class"].value_counts(normalize=True).sort_index())
print("Class distribution (test):")
print(test_df["perf_class"].value_counts(normalize=True).sort_index())

# 5. Feature selection: drop id/date/targets
drop_cols = [GROUP_COL, DATE_COL, TARGET_RET, "target_up", "nav_next", "_is_test", "perf_class"]
drop_cols = [c for c in drop_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in drop_cols]

# Remove very high missing or near-constant features based on train set
miss_frac = train_df[feature_cols].isnull().mean()
feature_cols = [c for c in feature_cols if miss_frac.get(c,0) <= 0.8]
stds = train_df[feature_cols].std(numeric_only=True)
feature_cols = [c for c in feature_cols if (not pd.isna(stds.get(c))) and stds.get(c) > 1e-8]

print("Using feature count:", len(feature_cols))

X_train = train_df[feature_cols].copy()
y_train = train_df["perf_class"].astype(int).copy()
X_test  = test_df[feature_cols].copy()
y_test  = test_df["perf_class"].astype(int).copy()

# 6. Imputer + scaler (logistic needs scaling; RF doesn't but pipeline keeps consistency)
imputer = SimpleImputer(strategy="median")

# 7. Baseline: Logistic Regression (with class_weight balanced)
lr_pipe = Pipeline([
    ("imputer", imputer),
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
])
print("\nTraining Logistic Regression baseline...")
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
print("Logistic Regression performance:")
print(classification_report(y_test, y_pred_lr, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_lr))

# 8. RandomForest classifier with small randomized search (safe parallelization)
rf_pipe = Pipeline([
    ("imputer", imputer),
    ("rf", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_jobs=8))
])
rf_param_dist = {
    "rf__n_estimators": [100, 150, 200],
    "rf__max_depth": [6, 8, 12, None],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4]
}
tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
rf_search = RandomizedSearchCV(
    rf_pipe, rf_param_dist, n_iter=min(RANDOM_SEARCH_ITERS, 12),
    cv=tscv, scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1, verbose=1
)
print("\nRunning RandomizedSearchCV for RandomForest (quick search)...")
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)
best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
print("RandomForest performance:")
print(classification_report(y_test, y_pred_rf, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Macro F1 RF:", f1_score(y_test, y_pred_rf, average="macro"), "Accuracy:", accuracy_score(y_test, y_pred_rf))

# 9. Optional: XGBoost classifier (if installed)
best_xgb = None
if xgb_installed:
    print("\nXGBoost available — running a small XGBoost trial (optional)...")
    xgb_pipe = Pipeline([
        ("imputer", imputer),
        ("xgb", xgb.XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='mlogloss'))
    ])
    xgb_param = {
        "xgb__n_estimators": [100, 200],
        "xgb__max_depth": [4, 6],
        "xgb__learning_rate": [0.05, 0.1],
        "xgb__subsample": [0.7, 1.0]
    }
    xgb_search = RandomizedSearchCV(
        xgb_pipe, xgb_param, n_iter=6, cv=tscv, scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1, verbose=1
    )
    xgb_search.fit(X_train, y_train)
    best_xgb = xgb_search.best_estimator_
    y_pred_xgb = best_xgb.predict(X_test)
    print("XGBoost performance:")
    print(classification_report(y_test, y_pred_xgb, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_xgb))

# 10. Pick best model by macro-F1 on test
candidates = {"Logistic": (lr_pipe, lr_pipe.predict(X_test))}
candidates["RandomForest"] = (best_rf, best_rf.predict(X_test))
if best_xgb is not None:
    candidates["XGBoost"] = (best_xgb, best_xgb.predict(X_test))

best_name = None
best_f1 = -1
for name, (model_obj, preds) in candidates.items():
    f1 = f1_score(y_test, preds, average="macro")
    if f1 > best_f1:
        best_f1 = f1
        best_name = name
print(f"\nBest model by macro-F1 on test: {best_name} (Macro-F1={best_f1:.4f})")

# 11. Save best model & feature list
final_model = candidates[best_name][0]
joblib.dump(final_model, os.path.join(OUT_DIR, f"classif_{best_name}.joblib"))
joblib.dump(feature_cols, os.path.join(OUT_DIR, "classif_feature_list.joblib"))
print("Saved model and feature list to", OUT_DIR)

# 12. Quick feature importance / coef summary for interpretability
print("\nTop features by Logistic coefficients (abs) -- if logistic chosen:")
if hasattr(lr_pipe.named_steps['lr'], "coef_"):
    coefs = lr_pipe.named_steps['lr'].coef_
    # For multiclass, coef_ is (n_classes, n_features) — take mean absolute across classes
    mean_abs = np.mean(np.abs(coefs), axis=0)
    feat_coef = pd.Series(mean_abs, index=X_train.columns).sort_values(ascending=False)
    print(feat_coef.head(10))

if best_name == "RandomForest":
    print("\nTop features by RandomForest importance:")
    rf_impl = final_model.named_steps['rf'] if hasattr(final_model, 'named_steps') else final_model
    if hasattr(rf_impl, "feature_importances_"):
        imp = pd.Series(rf_impl.feature_importances_, index=X_train.columns).sort_values(ascending=False)
        print(imp.head(10))

print("\nDone. Inspect printed metrics and saved artifacts in", OUT_DIR)


Loaded: (319086, 55)
Train rows: 255407 Test rows: 63679
Training quantiles -> 30%: -0.002289, 70%: 0.005207
Class distribution (train):
perf_class
0.0    0.300069
1.0    0.399930
2.0    0.300002
Name: proportion, dtype: float64
Class distribution (test):
perf_class
0    0.339484
1    0.422007
2    0.238509
Name: proportion, dtype: float64
Using feature count: 44

Training Logistic Regression baseline...
Logistic Regression performance:
              precision    recall  f1-score   support

           0     0.4530    0.1735    0.2509     21618
           1     0.4687    0.7573    0.5791     26873
           2     0.3433    0.2709    0.3029     15188

    accuracy                         0.4431     63679
   macro avg     0.4217    0.4006    0.3776     63679
weighted avg     0.4335    0.4431    0.4018     63679

Confusion matrix:
 [[ 3750 14006  3862]
 [ 2514 20350  4009]
 [ 2015  9058  4115]]

Running RandomizedSearchCV for RandomForest (quick search)...
Fitting 3 folds for each of 12 c